## Real Data Prediction Performance

This notebook loads saved prediction results for the real-data experiments. CKDR uses the final SCA setting and is loaded from per-run pickle files. Other methods are loaded from one pickle file per dataset and method.


In [1]:
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if not Path('results').exists() and Path('..', 'results').exists():
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))

try:
    import torch
except ModuleNotFoundError:
    torch = None

MASTER_SEED = 20241225
CKDR_DIR = Path('results/realdata_prediction/ckdr')
METHOD_DIR = Path('results/realdata_prediction')
TABLE_DIR = Path('results/tables')
TABLE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)


## Prediction Scripts

To reproduce the CKDR prediction pkl files, run these scripts from the repository root:

```bash
python reproducibility/realdata_prediction/CKDR_prediction.py --dataset gevers --dim-setting 3 --n_jobs -2
python reproducibility/realdata_prediction/CKDR_prediction.py --dataset gevers --dim-setting 5 --n_jobs -2
python reproducibility/realdata_prediction/CKDR_prediction.py --dataset gevers --dim-setting 3-7 --n_jobs -2
python reproducibility/realdata_prediction/CKDR_prediction.py --dataset ravel --dim-setting 3 --n_jobs -2
python reproducibility/realdata_prediction/CKDR_prediction.py --dataset ravel --dim-setting 5 --n_jobs -2
python reproducibility/realdata_prediction/CKDR_prediction.py --dataset ravel --dim-setting 3-7 --n_jobs -2
```

To reproduce the competitors' result pkl files, run:

```bash
python reproducibility/realdata_prediction/clr_Kernel_prediction.py
python reproducibility/realdata_prediction/clr_RF_prediction.py
python reproducibility/realdata_prediction/LC_Lasso_prediction.py --n_jobs -2
```

In [2]:
def load_pickle(path):
    path = Path(path)
    if torch is None and CKDR_DIR in path.parents:
        raise ImportError('CKDR pickle files require torch. Run this notebook in the project environment with torch installed.')
    with path.open('rb') as f:
        return pickle.load(f)


def load_ckdr_values(dataset, dim_setting, metric_key):
    values = []
    params = []
    for run in range(100):
        file = CKDR_DIR / f'{dataset}_dim{dim_setting}_run{run:03d}.pkl'
        record = load_pickle(file)
        values.append(record[metric_key])
        params.append(record['parameters'])
    return np.asarray(values, dtype=float), pd.DataFrame(params)


def load_method_values(dataset, method_file, expected_name):
    file = METHOD_DIR / method_file
    if not file.exists():
        print(f'Missing {file}. Run the corresponding script before final table export.')
        return np.full(100, np.nan)

    record = load_pickle(file)
    if record['dataset'] != dataset:
        raise ValueError(f'{file} stores dataset={record["dataset"]}, expected {dataset}')
    if record['method'] != expected_name:
        raise ValueError(f'{file} stores method={record["method"]}, expected {expected_name}')
    return np.asarray(record['values'], dtype=float)


def mean_se(values):
    values = pd.Series(values, dtype=float).dropna()
    return values.mean(), values.std(ddof=1) / np.sqrt(len(values))


def format_mean_se(values, scale=1.0, digits=2):
    mean, se = mean_se(np.asarray(values, dtype=float) * scale)
    return f'{mean:.{digits}f} ({se:.{digits}f})'


# Ileum Microbiome Data

Metric: classification accuracy for the raw result dataframe. The paper-style summary below reports misclassification percentage.


In [3]:
results_df = pd.DataFrame(index=np.arange(100))
ileum_params = {}

for dim_setting, column in [('3', 'CKDR-3'), ('5', 'CKDR-5'), ('3-7', 'CKDR')]:
    values, params = load_ckdr_values('ileum', dim_setting, 'test_acc')
    results_df[column] = values
    ileum_params[column] = params

results_df['LC-Lasso'] = load_method_values('ileum', 'ileum_LC-Lasso.pkl', 'LC-Lasso')
results_df['clr-Kernel (SVM)'] = load_method_values('ileum', 'ileum_clr-Kernel.pkl', 'clr-Kernel (SVM)')
results_df['clr-RF'] = load_method_values('ileum', 'ileum_clr-RF.pkl', 'clr-RF')

results_df.head()


,CKDR-3,CKDR-5,CKDR,LC-Lasso,clr-Kernel (SVM),clr-RF
0,0.750000,0.642857,0.642857,0.571429,0.535714,0.535714
1,0.678571,0.821429,0.678571,0.678571,0.750000,0.571429
2,0.714286,0.714286,0.607143,0.678571,0.750000,0.642857
3,0.642857,0.607143,0.642857,0.785714,0.678571,0.714286
4,0.678571,0.678571,0.678571,0.607143,0.642857,0.642857


In [4]:
ileum_mcr_summary = pd.DataFrame(
    [[format_mean_se(1 - results_df[col], scale=100, digits=1) for col in results_df.columns]],
    index=['MCR (\%)'],
    columns=results_df.columns,
)

ileum_mcr_summary


,CKDR-3,CKDR-5,CKDR,LC-Lasso,clr-Kernel (SVM),clr-RF
MCR (\%),26.3 (0.8),25.5 (0.8),26.3 (0.8),28.5 (0.7),28.2 (0.7),34.5 (0.9)


### Ileum Paired t-tests

Positive `mean_mcr_improvement` means the CKDR method has lower misclassification rate than the competitor.


In [5]:
from scipy.stats import ttest_1samp

comparisons = []
for ckdr_col in ['CKDR-3', 'CKDR-5']:
    ckdr_mcr = 1 - results_df[ckdr_col]
    for competitor in ['LC-Lasso', 'clr-Kernel (SVM)']:
        competitor_mcr = 1 - results_df[competitor]
        valid = pd.concat([ckdr_mcr, competitor_mcr], axis=1).dropna()
        if len(valid) == 0:
            comparisons.append({
                'ckdr_method': ckdr_col,
                'competitor': competitor,
                'n': 0,
                'mean_mcr_improvement': np.nan,
                't_stat': np.nan,
                'p_one_sided': np.nan,
                'significant_0.05': False,
            })
            continue

        diff = valid.iloc[:, 1] - valid.iloc[:, 0]
        test = ttest_1samp(diff, popmean=0.0, alternative='greater')
        comparisons.append({
            'ckdr_method': ckdr_col,
            'competitor': competitor,
            'n': len(diff),
            'mean_mcr_improvement': diff.mean(),
            't_stat': test.statistic,
            'p_one_sided': test.pvalue,
            'significant_0.05': test.pvalue < 0.05,
        })

ileum_ttest = pd.DataFrame(comparisons)
ileum_ttest['mean_mcr_improvement'] *= 100
ileum_ttest


,ckdr_method,competitor,n,mean_mcr_improvement,t_stat,p_one_sided,significant_0.05
0,CKDR-3,LC-Lasso,100,2.142857,2.168374,0.016263,True
1,CKDR-3,clr-Kernel (SVM),100,1.857143,2.086400,0.019756,True
2,CKDR-5,LC-Lasso,100,2.964286,3.264137,0.000754,True
3,CKDR-5,clr-Kernel (SVM),100,2.678571,3.507112,0.000341,True


# Vaginal Microbiome Data

Metric: mean squared error. Lower is better.


In [6]:
results_df2 = pd.DataFrame(index=np.arange(100))
vaginal_params = {}

for dim_setting, column in [('3', 'CKDR-3'), ('5', 'CKDR-5'), ('3-7', 'CKDR')]:
    values, params = load_ckdr_values('vaginal', dim_setting, 'test_metric')
    results_df2[column] = values
    vaginal_params[column] = params

results_df2['LC-Lasso'] = load_method_values('vaginal', 'vaginal_LC-Lasso.pkl', 'LC-Lasso')
results_df2['clr-Kernel (KRR)'] = load_method_values('vaginal', 'vaginal_clr-Kernel.pkl', 'clr-Kernel (KRR)')
results_df2['clr-RF'] = load_method_values('vaginal', 'vaginal_clr-RF.pkl', 'clr-RF')


rs_es_file = Path('reproducibility/other_methods/Relative-shift/JP_realdata_100reps.mat')
if rs_es_file.exists():
    from scipy.io import loadmat
    results_df2['RS-ES'] = loadmat(rs_es_file)['result_test20'].ravel()
else:
    results_df2['RS-ES'] = np.nan

results_df2.head()


,CKDR-3,CKDR-5,CKDR,LC-Lasso,clr-Kernel (KRR),clr-RF,RS-ES
0,3.284516,3.394199,3.216210,2.758368,2.637225,2.885940,2.036339
1,1.949667,1.961172,1.960659,3.792503,3.318649,3.983409,4.571289
2,2.905773,2.900604,2.902707,2.727516,2.558037,1.895928,3.910931
3,2.160079,2.169020,2.160079,3.293235,3.047152,3.335659,3.373369
4,2.457253,2.456088,2.452850,3.271276,3.119127,3.630609,3.907272


In [7]:
vaginal_mse_summary = pd.DataFrame(
    [[format_mean_se(results_df2[col], scale=1.0, digits=2) for col in results_df2.columns]],
    index=['MSE'],
    columns=results_df2.columns,
)

vaginal_mse_summary


,CKDR-3,CKDR-5,CKDR,LC-Lasso,clr-Kernel (KRR),clr-RF,RS-ES
MSE,3.27 (0.08),3.26 (0.08),3.27 (0.08),3.76 (0.06),3.41 (0.06),3.31 (0.07),4.20 (0.09)


### Vaginal Paired t-tests

Positive `mean_mse_improvement` means the CKDR method has lower MSE than the competitor.


In [8]:
comparisons = []
for ckdr_col in ['CKDR-3', 'CKDR-5']:
    ckdr_mse = results_df2[ckdr_col]
    for competitor in ['LC-Lasso', 'clr-Kernel (KRR)', 'clr-RF', 'RS-ES']:
        competitor_mse = results_df2[competitor]
        valid = pd.concat([ckdr_mse, competitor_mse], axis=1).dropna()
        if len(valid) == 0:
            comparisons.append({
                'ckdr_method': ckdr_col,
                'competitor': competitor,
                'n': 0,
                'mean_mse_improvement': np.nan,
                't_stat': np.nan,
                'p_one_sided': np.nan,
                'significant_0.05': False,
            })
            continue

        diff = valid.iloc[:, 1] - valid.iloc[:, 0]
        test = ttest_1samp(diff, popmean=0.0, alternative='greater')
        comparisons.append({
            'ckdr_method': ckdr_col,
            'competitor': competitor,
            'n': len(diff),
            'mean_mse_improvement': diff.mean(),
            't_stat': test.statistic,
            'p_one_sided': test.pvalue,
            'significant_0.05': test.pvalue < 0.05,
        })

vaginal_ttest = pd.DataFrame(comparisons)
vaginal_ttest


,ckdr_method,competitor,n,mean_mse_improvement,t_stat,p_one_sided,significant_0.05
0,CKDR-3,LC-Lasso,100,0.495376,5.079063,8.932636e-07,True
1,CKDR-3,clr-Kernel (KRR),100,0.145714,1.473809,7.185285e-02,False
2,CKDR-3,clr-RF,100,0.040426,0.397726,3.458441e-01,False
3,CKDR-3,RS-ES,100,0.931271,8.407209,1.597378e-13,True
4,CKDR-5,LC-Lasso,100,0.505115,5.173808,6.004277e-07,True
5,CKDR-5,clr-Kernel (KRR),100,0.155453,1.580145,5.863173e-02,False
6,CKDR-5,clr-RF,100,0.050165,0.492457,3.117432e-01,False
7,CKDR-5,RS-ES,100,0.941010,8.441318,1.348161e-13,True


## LaTeX Table Export

The exported table combines ileum and vaginal prediction results. Ileum reports misclassification rate (MCR, %), and vaginal reports MSE. The best value in each row is bolded.


In [9]:
def _mean_se(values, transform):
    vals = transform(pd.Series(values, dtype=float).dropna().to_numpy())
    if len(vals) == 0:
        return np.nan, np.nan
    return vals.mean(), vals.std(ddof=1) / np.sqrt(len(vals))


def _format_stat(mean, se, digits, bold=False):
    if pd.isna(mean):
        return '--'
    entry = f'{mean:.{digits}f} ({se:.{digits}f})'
    return rf'\textbf{{{entry}}}' if bold else entry


def _best_method(stats):
    valid = {method: value for method, value in stats.items() if not pd.isna(value[0])}
    return min(valid, key=lambda method: valid[method][0])


methods = ['CKDR-3', 'CKDR-5', 'CKDR', 'LC-Lasso', 'clr-Kernel', 'clr-RF', 'RS-ES']
header = ['CKDR-3', 'CKDR-5', 'CKDR$^*$', 'LC-Lasso', 'clr-Kernel', 'clr-RF', 'RS-ES']

ileum_source = {
    'CKDR-3': results_df['CKDR-3'],
    'CKDR-5': results_df['CKDR-5'],
    'CKDR': results_df['CKDR'],
    'LC-Lasso': results_df['LC-Lasso'],
    'clr-Kernel': results_df['clr-Kernel (SVM)'],
    'clr-RF': results_df['clr-RF'],
}
ileum_stats = {
    method: _mean_se(values, lambda x: 100 * (1 - x))
    for method, values in ileum_source.items()
}
ileum_best = _best_method(ileum_stats)

vaginal_source = {
    'CKDR-3': results_df2['CKDR-3'],
    'CKDR-5': results_df2['CKDR-5'],
    'CKDR': results_df2['CKDR'],
    'LC-Lasso': results_df2['LC-Lasso'],
    'clr-Kernel': results_df2['clr-Kernel (KRR)'],
    'clr-RF': results_df2['clr-RF'],
    'RS-ES': results_df2['RS-ES'],
}
vaginal_stats = {
    method: _mean_se(values, lambda x: x)
    for method, values in vaginal_source.items()
}
vaginal_best = _best_method(vaginal_stats)

ileum_row = [
    _format_stat(*ileum_stats[method], digits=1, bold=(method == ileum_best))
    if method in ileum_stats else '--'
    for method in methods
]
vaginal_row = [
    _format_stat(*vaginal_stats[method], digits=2, bold=(method == vaginal_best))
    for method in methods
]

combined_table = pd.DataFrame(
    [ileum_row, vaginal_row],
    index=[r'Ileum MCR (\%)', 'Vaginal MSE'],
    columns=header,
)

header_line = ' & ' + ' & '.join(header) + r' \\'
ileum_line = r'Ileum MCR (\%) & ' + ' & '.join(ileum_row) + r' \\'
vaginal_line = 'Vaginal MSE & ' + ' & '.join(vaginal_row) + r' \\'

table_tex = (
    r'\begin{table}%[t]' + '\n'
    r'    \centering' + '\n'
    r'    \caption{Prediction performance for ileum and vaginal microbiome data. The ileum row reports misclassification rate (MCR, \%; standard errors in parentheses), and the vaginal row reports mean squared error (standard errors in parentheses).}' + '\n'
    r'    \label{tab: real prediction}' + '\n'
    r'    \small' + '\n'
    r'    \begin{tabular}{lccccccc}' + '\n'
    r'        \toprule' + '\n'
    f'        {header_line}' + '\n'
    r'        \midrule' + '\n'
    f'        {ileum_line}' + '\n'
    f'        {vaginal_line}' + '\n'
    r'        \bottomrule' + '\n'
    r'    \end{tabular}%' + '\n'
    r'\end{table}' + '\n'
)

out_file = TABLE_DIR / 'realdata_prediction.tex'
out_file.write_text(table_tex, encoding='utf-8')

print(f'Wrote {out_file}')
display(combined_table)


Wrote results\tables\realdata_prediction.tex


,CKDR-3,CKDR-5,CKDR$^*$,LC-Lasso,clr-Kernel,clr-RF,RS-ES
Ileum MCR (\%),26.3 (0.8),\textbf{25.5 (0.8)},26.3 (0.8),28.5 (0.7),28.2 (0.7),34.5 (0.9),--
Vaginal MSE,3.27 (0.08),\textbf{3.26 (0.08)},3.27 (0.08),3.76 (0.06),3.41 (0.06),3.31 (0.07),4.20 (0.09)
